# Low-level data access - the CAS format

In [1]:
from dakoda.corpus import DakodaCorpus, DakodaPublicCorpusName

wtld = DakodaCorpus(DakodaPublicCorpusName.MERL_L2)
doc = wtld.random_doc()
cas = doc.cas

# in the high-level API this is just doc.text
print(cas.sofa_string)  # the document text
print(len(list(cas.select('Token'))))  # number of tokens

# this view is empty!
# we need to get the learner text view
learner_view = cas.get_view('ctok')

# print all tokens in the learner view with their offsets
for token in learner_view.select('Token'):
    print(token.begin, token.end, token.get_covered_text())

[CACHE LOAD] Using cached corpus: MERL-L2
[LOCAL LOAD] Loading corpus from: /Users/zesch/git/dakoda/analysis-recipes/.dakoda/corpora/MERL-L2
Stadt X , 08.03.2012 Lieber Jens , wie geht es dir ? Glückwünsche für dich , jetzt bist du Vater geworden . Ich hoffe dich gut machen . Wie geht es Sibylle ? Muss sie im Kankenhaus geblieben ? Hast du zwei Babys ? Wie heißen ihre Babys ? Hast du eine Party gemacht ? Ich warte deine Anworten . Viele Grüße Meier
0
0 5 Stadt
6 7 X
8 9 ,
10 20 08.03.2012
21 27 Lieber
28 32 Jens
33 34 ,
35 38 wie
39 43 geht
44 46 es
47 50 dir
51 52 ?
53 65 Glückwünsche
66 69 für
70 74 dich
75 76 ,
77 82 jetzt
83 87 bist
88 90 du
91 96 Vater
97 105 geworden
106 107 .
108 111 Ich
112 117 hoffe
118 122 dich
123 126 gut
127 133 machen
134 135 .
136 139 Wie
140 144 geht
145 147 es
148 155 Sibylle
156 157 ?
158 162 Muss
163 166 sie
167 169 im
170 180 Kankenhaus
181 190 geblieben
191 192 ?
193 197 Hast
198 200 du
201 205 zwei
206 211 Babys
212 213 ?
214 217 Wie
218 224 heißen

In [2]:
from collections import Counter

# get most frequent finite verbs
T_POS = 'de.tudarmstadt.ukp.dkpro.core.api.lexmorph.type.pos.POS'
T_LEMMA = 'de.tudarmstadt.ukp.dkpro.core.api.segmentation.type.Lemma'
finite_verb_tags = ['VAFIN','VVFIN','VMFIN']
verbs = Counter()

for doc in wtld:
    cas = doc.cas
    v = cas.get_view('ctok')

    for t in v.select('Token'):
        pos = v.select_covered(type_=T_POS, covering_annotation=t)[0]
        lemma = v.select_covered(type_=T_LEMMA, covering_annotation=t)[0]
        if pos.PosValue in finite_verb_tags:
            verbs[lemma.value] += 1

# get top 20 most frequent
for verb, freq in verbs.most_common(20):
    print(verb, freq)

sein 3170
haben 2287
können 1320
mögen 800
werden 605
muss 447
gehen 424
geben 334
sollen 313
wollen 259
brauchen 204
kommen 185
finden 160
suchen 148
wünschen 147
leben 142
freuen 126
hoffen 123
müssen 118
machen 117


In [3]:
# doppelt belegte Stage Annotationen
T_STAGE = 'org.dakoda.Stage'
T_SENTENCE = 'de.tudarmstadt.ukp.dkpro.core.api.segmentation.type.Sentence'
T_TOKEN = 'de.tudarmstadt.ukp.dkpro.core.api.segmentation.type.Token'
verbs = {}
for doc in wtld:
    cas = doc.cas
    v = cas.get_view('ctok')

    for token in v.select(T_TOKEN):
        stages = v.select_covered(type_=T_STAGE, covering_annotation=token)
        if len(stages) > 2:
            print(doc.learner.text)
            for s in stages:
                print(f"{stages[0].get_covered_text()} - {s.name}")

Friedrich Meier Müllergasse 1 12 - 123 Stadt X Computer-Spezialist Odenwaldstraße 5 53119 Bonn Stadt X , 07.07.2010 Bewerbung um ein Praktikum Sehr geehrte Damen und Herren , ich habe mit große Interesse Ihre Anzeige in der „ Computer " Zeitschrift vom 25.06.2010 gelesen . Sie suchen ein Vertriebspraktikant in Ihre Firma . Ich studierte Informatik an der Schlesischen Technischen Universität . Um mein Studium abzuschließen sollte ich noch ein Praktikum bei einer Firma im Ausland machen . Deswegen möchte ich mich um ein Praktikum bei Ihren bewerben . Während meines Studium hatte ich schon ein Praktikum bei einer polnischen Firma , in Verkaufabteilung , wo hatte ich die Möglichkeit , mit den ausländischen Kunden zu arbeiten . Jetzt suche ich ein Praktikum wo ich meine Sprachkenntni-unreadable- verbessern kann . Ich spreche auf englisch und auf deutsch . Ich denke , dass ich die Erfahrungen , die ich bei Ihnen sammeln kann , könnte ich in meiner zukunftigen Arbeit benutzen . Ich arbeite ge